# Customer Segmentation with K-Means
**Dataset:** [Customer Personality Analysis](https://www.kaggle.com/datasets/imakash3011/customer-personality-analysis) (`marketing_campaign.csv`, ~2,240 customers)

Steps:
1. Load the data with `kagglehub`
2. Clean it and engineer features (age, total spend, children, tenure...)
3. Scale the features
4. Choose the number of clusters (elbow method + silhouette score)
5. Fit K-Means
6. Visualise and profile each segment

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
RANDOM_STATE = 42

## 1. Load the data
The file is **tab-separated**, so `sep="\t"` is needed.

In [ ]:
path = kagglehub.dataset_download("imakash3011/customer-personality-analysis")
print("Downloaded to:", path)
print(os.listdir(path))

df_raw = pd.read_csv(os.path.join(path, "marketing_campaign.csv"), sep="\t")
print("Shape:", df_raw.shape)
df_raw.head()

In [ ]:
df_raw.info()
print("\nMissing values per column:")
print(df_raw.isna().sum()[df_raw.isna().sum() > 0])

## 2. Cleaning and feature engineering
- Drop rows with missing `Income` (only ~24 rows)
- Parse `Dt_Customer` (format `dd-mm-yyyy`) and compute how long each person has been a customer
- **Age** relative to the last enrolment date in the data (the data was collected around 2014, so using today's date would inflate every age)
- **Spent** = total spend across all 6 product categories
- **Children**, **Is_Parent**, **Family_Size**
- Simplify `Education` and `Marital_Status`
- **TotalPurchases** and **TotalAcceptedCmp** (number of campaigns accepted)
- Drop constant columns (`Z_CostContact`, `Z_Revenue`) and `ID`

In [ ]:
df = df_raw.copy()
df = df.dropna(subset=["Income"])

df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"], format="%d-%m-%Y")
ref_date = df["Dt_Customer"].max()
df["Customer_Days"] = (ref_date - df["Dt_Customer"]).dt.days
df["Age"] = ref_date.year - df["Year_Birth"]

mnt_cols = ["MntWines", "MntFruits", "MntMeatProducts",
            "MntFishProducts", "MntSweetProducts", "MntGoldProds"]
purchase_cols = ["NumWebPurchases", "NumCatalogPurchases", "NumStorePurchases"]
cmp_cols = ["AcceptedCmp1", "AcceptedCmp2", "AcceptedCmp3", "AcceptedCmp4", "AcceptedCmp5"]

df["Spent"] = df[mnt_cols].sum(axis=1)
df["TotalPurchases"] = df[purchase_cols].sum(axis=1)
df["TotalAcceptedCmp"] = df[cmp_cols].sum(axis=1)

df["Children"] = df["Kidhome"] + df["Teenhome"]
df["Is_Parent"] = (df["Children"] > 0).astype(int)

df["Living_With"] = df["Marital_Status"].replace({
    "Married": "Partner", "Together": "Partner",
    "Single": "Alone", "Divorced": "Alone", "Widow": "Alone",
    "Absurd": "Alone", "YOLO": "Alone", "Alone": "Alone",
})
df["Has_Partner"] = (df["Living_With"] == "Partner").astype(int)
df["Family_Size"] = 1 + df["Has_Partner"] + df["Children"]

# Ordinal education level: 0 = Basic, 1 = Graduate (incl. 2n Cycle), 2 = Postgraduate
df["Education_Level"] = df["Education"].map({
    "Basic": 0, "2n Cycle": 1, "Graduation": 1, "Master": 2, "PhD": 2,
}).fillna(1).astype(int)

df = df.drop(columns=["ID", "Z_CostContact", "Z_Revenue"], errors="ignore")
print("Shape after cleaning:", df.shape)
df[["Age", "Income", "Spent", "Children", "Customer_Days", "TotalPurchases"]].describe()

### Outliers
The raw data contains a few impossible ages (birth years in the 1890s/1900s) and one income of 666,666.
These distort K-Means (which uses distances, so extreme points pull centroids), so we remove them.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(x=df["Age"], ax=axes[0]).set_title("Age")
sns.boxplot(x=df["Income"], ax=axes[1]).set_title("Income")
plt.tight_layout()
plt.show()

before = len(df)
df = df[(df["Age"] < 90) & (df["Income"] < 600_000)].reset_index(drop=True)
print(f"Removed {before - len(df)} outlier rows -> {len(df)} customers remain")

## 3. Select and scale features
K-Means is distance-based, so every feature must be on the same scale (`StandardScaler`: mean 0, std 1).
We leave out the campaign response columns (`AcceptedCmp*`, `Response`, `Complain`) so we can use them
afterwards to check how the segments behave.

In [ ]:
features = [
    "Income", "Age", "Education_Level", "Has_Partner", "Children", "Family_Size", "Is_Parent",
    "Customer_Days", "Recency", "Spent", *mnt_cols,
    "NumDealsPurchases", *purchase_cols, "NumWebVisitsMonth",
]
X = df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("Feature matrix:", X_scaled.shape)

## 4. Choose the number of clusters
- **Elbow method:** inertia (within-cluster sum of squares) for k = 2..10. Look for the bend.
- **Silhouette score:** how well separated the clusters are (-1 to 1, higher is better).

In [ ]:
k_range = range(2, 11)
inertias, silhouettes = [], []
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(list(k_range), inertias, "o-")
axes[0].set(title="Elbow method", xlabel="k (number of clusters)", ylabel="Inertia")
axes[1].plot(list(k_range), silhouettes, "o-", color="tab:orange")
axes[1].set(title="Silhouette score", xlabel="k (number of clusters)", ylabel="Silhouette")
plt.tight_layout()
plt.show()

pd.DataFrame({"k": list(k_range), "inertia": inertias, "silhouette": silhouettes}).round(3)

**Pick K.** k = 2 often scores best on silhouette here, but it usually only splits customers into
"high spenders" and "low spenders", which isn't very useful for marketing. **k = 4** is a common choice for this
dataset: it sits near the elbow and gives segments you can act on.
Use the plots above and change `K` if your results point elsewhere.

In [ ]:
K = 4

kmeans = KMeans(n_clusters=K, n_init=10, random_state=RANDOM_STATE)
df["Cluster"] = kmeans.fit_predict(X_scaled)

print(f"Silhouette score for k={K}: {silhouette_score(X_scaled, df['Cluster']):.3f}")
df["Cluster"].value_counts().sort_index()

## 5. Visualise the clusters
PCA reduces the 21 scaled features to 2 dimensions **for plotting only**. The clustering itself was done on all features.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
centers_pca = pca.transform(kmeans.cluster_centers_)
print("Variance explained by 2 components:", pca.explained_variance_ratio_.sum().round(3))

palette = sns.color_palette("tab10", K)
plt.figure(figsize=(9, 6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=df["Cluster"], palette=palette, s=25, alpha=0.7)
plt.scatter(centers_pca[:, 0], centers_pca[:, 1], c="black", marker="X", s=200, label="Centroids")
plt.title("Customer clusters (PCA projection)")
plt.xlabel("PC1"); plt.ylabel("PC2")
plt.legend(title="Cluster")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(x="Cluster", data=df, palette=palette, hue="Cluster", legend=False, ax=axes[0])
axes[0].set_title("Customers per cluster")
sns.scatterplot(x="Income", y="Spent", hue="Cluster", data=df, palette=palette, s=25, alpha=0.7, ax=axes[1])
axes[1].set_title("Income vs total spend")
plt.tight_layout()
plt.show()

In [ ]:
profile_vars = ["Age", "Children", "Customer_Days", "TotalPurchases", "NumDealsPurchases", "NumWebVisitsMonth"]
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flat, profile_vars):
    sns.boxplot(x="Cluster", y=col, data=df, palette=palette, hue="Cluster", legend=False, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 6. Profile the segments
Average values per cluster, plus how often each segment responded to marketing campaigns
(these columns were **not** used for clustering, so they show whether the segments differ in behaviour).

In [ ]:
profile = df.groupby("Cluster").agg(
    Customers=("Income", "size"),
    Income=("Income", "mean"),
    Age=("Age", "mean"),
    Spent=("Spent", "mean"),
    Children=("Children", "mean"),
    Is_Parent=("Is_Parent", "mean"),
    Has_Partner=("Has_Partner", "mean"),
    TotalPurchases=("TotalPurchases", "mean"),
    DealsPurchases=("NumDealsPurchases", "mean"),
    WebVisits=("NumWebVisitsMonth", "mean"),
    Campaigns_Accepted=("TotalAcceptedCmp", "mean"),
    Response_Rate=("Response", "mean"),
).round(2)
profile

In [ ]:
# Share of spend per product category, per cluster
spend_share = df.groupby("Cluster")[mnt_cols].sum()
spend_share = spend_share.div(spend_share.sum(axis=1), axis=0)
spend_share.columns = [c.replace("Mnt", "").replace("Products", "").replace("Prods", "") for c in spend_share.columns]
spend_share.plot(kind="bar", stacked=True, figsize=(9, 5), colormap="tab20")
plt.title("Share of spend by product category")
plt.ylabel("Share of total spend")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Interpreting the clusters
Look at the profile table above and give each cluster a name. Patterns people often find in this dataset (check yours against the table):
- **High income, high spend, mostly no children:** premium customers who respond best to campaigns
- **Low income, low spend, young children:** budget-conscious, many web visits, buy on deals
- **Middle income, teenagers at home:** moderate spend, heavy deal users
- **Older, higher income parents:** solid spenders, lean towards wine and store purchases

Cluster numbers are arbitrary and can change between runs or versions of scikit-learn, so always match names to the profile table.

In [ ]:
df.to_csv("customer_segments.csv", index=False)
print("Saved customer_segments.csv with a 'Cluster' column")